# Level 4 — 심층 분석: Grad-CAM 및 효율성 Trade-off

**목표**: 모델의 *동작 원리* 를 설명하고, FPS와 정확도의 trade-off 를 정량화합니다.

리포트 필수 산출물:
1. **속성별 정규화 Confusion Matrix 3개** — best 모델 기준.
2. **Grad-CAM 패널** — 같은 이미지에 대해 3개 head 가 각각 어디를 보는지 시각화 (예: `rainy + night + city street` 인 이미지).
3. 모든 백본에 대한 **FPS vs Avg-MF1 Pareto plot**.

본 노트북은 학습 노트북이 아니라 분석 노트북이지만, wandb 가 활성화되어 있으면 confusion matrix 이미지·Grad-CAM 패널·FPS 표를 같은 프로젝트의 별도 Run 으로 업로드합니다.

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/IRCVLab/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed
from src.utils.transforms import eval_transform
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, average_macro_f1, CLASS_NAMES
from src.utils.efficiency import measure_fps
from src.utils.wandb_logger import WandbLogger
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.xai.gradcam import GradCAM
from src.models.resnet import resnet18, resnet50

set_seed(42, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb

# API key 로그인 (최초 1회 입력하면 이후 캐시됨). WANDB_DISABLED=true 이면 건너뛴다.
if os.environ.get("WANDB_DISABLED", "").lower() != "true":
    wandb.login(key="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")

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
logger = WandbLogger(project=WANDB_PROJECT, run_name="level4-analysis", tags=["level4", "analysis"])

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# 분석 대상으로 사용할 best 모델 로드 (예시: ResNet-50)
val_ds = BDDAttrDataset("../data/set_a", "val", transform=eval_transform())
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

model = resnet50().to(device)
ckpt = torch.load("../checkpoints/level1_resnet50.pth", map_location=device)
model.load_state_dict(ckpt["state_dict"])

In [ ]:
# 속성별 정규화 Confusion Matrix 생성 및 시각화
preds, probs, targets, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(preds, targets, normalize="true")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, a in zip(axes, ATTRIBUTES):
    sns.heatmap(cms[a], annot=True, fmt=".2f", cmap="Blues", ax=ax, cbar=False,
                xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a])
    ax.set_title(f"{a}")
    ax.set_xlabel("예측 (pred)"); ax.set_ylabel("정답 (true)")
fig.tight_layout()
logger.log_image("analysis/confusion_matrices", fig)

# 속성별 개별 confusion matrix 도 분리해서 업로드
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"analysis/cm_{a}", cms[a], CLASS_NAMES[a])

In [ ]:
# 속성별 Grad-CAM. ResNet 의 경우 마지막 conv 단계를 target_layer 로 사용.
target_layer = model.layer4[-1]
gc = GradCAM(model, target_layer)

batch = next(iter(val_loader))
x = batch["image"][:6].to(device)  # 샘플 이미지 6장

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for row, attr in enumerate(ATTRIBUTES):
    # 각 속성 head 의 최대 logit 합을 score 로 사용 → 해당 head 가 "보는" 영역 추출
    cam = gc(x, lambda out, a=attr: out[a].max(dim=-1).values.sum())
    for col in range(6):
        img = x[col].cpu().permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min())
        axes[row, col].imshow(img)
        axes[row, col].imshow(cam[col].cpu().numpy(), cmap="jet", alpha=0.45)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(attr, fontsize=14)
fig.tight_layout()
logger.log_image("analysis/gradcam_panel", fig)

In [ ]:
# FPS 측정 + FPS vs Avg-MF1 Pareto plot.
# FPS 는 구조에만 의존하므로 random-init 으로도 측정 가능하지만, Avg-MF1 은
# 학습된 체크포인트가 필요하다. 존재하는 체크포인트만 평가에 포함한다.
import os
from src.models.vgg import VGG16
from src.models.vit import vit_small_patch16_224

backbones = [
    ("vgg16",    VGG16,                  "../checkpoints/level1_vgg16.pth"),
    ("resnet18", resnet18,               "../checkpoints/level1_resnet18.pth"),
    ("resnet50", resnet50,               "../checkpoints/level1_resnet50.pth"),
    ("vit_s16",  vit_small_patch16_224,  "../checkpoints/level2_vit_s16.pth"),
]

fps_rows, pareto = [], []
for fn, name, ckpt in [(b[1], b[0], b[2]) for b in backbones]:
    m = fn().to(device).eval()
    fps = measure_fps(m, device, batch_size=1, n_warmup=20, n_iter=200)
    fps_rows.append([name, round(fps, 2)])

    if os.path.exists(ckpt):                       # 학습된 가중치가 있으면 Avg-MF1 도 계산
        state = torch.load(ckpt, map_location=device)
        m.load_state_dict(state["state_dict"] if "state_dict" in state else state)
        preds_b, _, tgts_b, _ = collect_predictions(m, val_loader, device)
        mf1 = average_macro_f1(preds_b, tgts_b)
        pareto.append((name, fps, mf1))
        print(f"{name:10s} FPS={fps:7.2f}  Avg-MF1={mf1:.4f}")
    else:
        print(f"{name:10s} FPS={fps:7.2f}  (checkpoint 없음 → Pareto 제외)")

logger.log_table("analysis/fps", ["backbone", "FPS"], fps_rows)

# FPS (x) vs Avg-MF1 (y) Pareto plot
if pareto:
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, fps, mf1 in pareto:
        ax.scatter(fps, mf1, s=90)
        ax.annotate(name, (fps, mf1), textcoords="offset points", xytext=(6, 4), fontsize=11)
    ax.set_xlabel("FPS (batch=1)"); ax.set_ylabel("Avg-Macro-F1")
    ax.set_title("Efficiency vs Accuracy — Pareto")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    logger.log_image("analysis/pareto", fig)
else:
    print("Pareto plot 생략 — 학습된 체크포인트가 하나도 없습니다. Level 1/2 를 먼저 실행하세요.")

In [ ]:
logger.finish()